# XGBoost Training

This notebook loads the prepared scaled train and test parquet files from `../data/scaled/`, trains the `xgboost` model, saves the fitted model to `../model/`, and appends run history to `../model/results/`.


## 1. Imports and Config

Set the shared file locations, output paths, run metadata, and notebook-local helpers for this training run.


In [ ]:
from datetime import datetime
from pathlib import Path
import json

import polars as pl
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

DATA_DIR = Path("..") / "data" / "scaled"
X_TRAIN_FILE = DATA_DIR / "X_train.parquet"
Y_TRAIN_FILE = DATA_DIR / "y_train.parquet"
X_TEST_FILE = DATA_DIR / "X_test.parquet"
Y_TEST_FILE = DATA_DIR / "y_test.parquet"
MODEL_DIR = Path("..") / "model"
RESULTS_DIR = MODEL_DIR / "results"
TARGET_COLUMN = "label"
RANDOM_SEED = 42
MODEL_NAME = "xgboost"
RUN_AT = datetime.now().astimezone()
RUN_ID = RUN_AT.strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = RUN_AT.isoformat(timespec="seconds")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / "xgboost_model.json"
RESULTS_FILE = RESULTS_DIR / "xgboost_results.csv"
METRICS_FILE = RESULTS_DIR / "xgboost_metrics.csv"


def append_frame_to_csv(frame: pl.DataFrame, path: Path) -> None:
    if path.exists():
        existing = pl.read_csv(path)
        frame = pl.concat([existing, frame], how="diagonal_relaxed")
    frame.write_csv(path)


## 2. Load the Dataset

Load the prepared feature matrices and target vectors from the scaled data folder.


In [ ]:
for required_file in [X_TRAIN_FILE, Y_TRAIN_FILE, X_TEST_FILE, Y_TEST_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required parquet file not found: {required_file}")

X_train = pl.read_parquet(X_TRAIN_FILE)
y_train_df = pl.read_parquet(Y_TRAIN_FILE)
X_test = pl.read_parquet(X_TEST_FILE)
y_test_df = pl.read_parquet(Y_TEST_FILE)

print(f"loaded X_train: {X_train.shape}")
print(f"loaded y_train: {y_train_df.shape}")
print(f"loaded X_test: {X_test.shape}")
print(f"loaded y_test: {y_test_df.shape}")


## 3. Separate Features and Target

Keep the feature matrices separate and extract the binary target from the `label` column in the target parquet files.


In [ ]:
y_train = y_train_df.get_column(TARGET_COLUMN).cast(pl.Int32)
y_test = y_test_df.get_column(TARGET_COLUMN).cast(pl.Int32)

print(f"feature columns: {len(X_train.columns)}")
print(f"target column: {TARGET_COLUMN}")


## 4. Validate the Prepared Features

Fail fast if the upstream split and scaling pipeline did not produce aligned numeric model inputs.


In [ ]:
for frame_name, frame in {"y_train": y_train_df, "y_test": y_test_df}.items():
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(f"{frame_name}.parquet must include the target column: {TARGET_COLUMN}")
    if frame.width != 1:
        raise ValueError(f"{frame_name}.parquet should only contain the target column: {TARGET_COLUMN}")

if TARGET_COLUMN in X_train.columns or TARGET_COLUMN in X_test.columns:
    raise ValueError(f"X parquet files should only contain features, not the target column: {TARGET_COLUMN}")

if X_train.columns != X_test.columns:
    raise ValueError("X_train and X_test must contain the same feature columns in the same order.")

if X_train.height != y_train.len():
    raise ValueError("X_train and y_train must have the same number of rows.")

if X_test.height != y_test.len():
    raise ValueError("X_test and y_test must have the same number of rows.")

non_numeric_train = [column_name for column_name, dtype in X_train.schema.items() if not dtype.is_numeric()]
non_numeric_test = [column_name for column_name, dtype in X_test.schema.items() if not dtype.is_numeric()]
non_numeric_features = sorted(set(non_numeric_train + non_numeric_test))

if non_numeric_features:
    preview = ", ".join(non_numeric_features[:10])
    raise ValueError(
        "The split/feature-selection notebook must output model-ready numeric features. "
        f"Non-numeric features found: {preview}"
    )

print(f"validated {len(X_train.columns)} numeric features")


## 5. Configure Hyperparameters

Use this cell to review and edit the XGBoost settings before training. The class-weight default is computed from the current training labels and can still be overridden here.


In [ ]:
positive_count = int(y_train.sum())
negative_count = y_train.len() - positive_count
default_scale_pos_weight = negative_count / positive_count if positive_count else 1.0
prediction_threshold = 0.5

hyperparameters = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "scale_pos_weight": default_scale_pos_weight,
}


hyperparameters_json = json.dumps(hyperparameters, sort_keys=True)
hyperparameters_frame = pl.DataFrame(
    [
        {
            "model": MODEL_NAME,
            "run_id": RUN_ID,
            "run_timestamp": RUN_TIMESTAMP,
            "prediction_threshold": prediction_threshold,
            "hyperparameters_json": hyperparameters_json,
        }
    ]
)

hyperparameters_frame


## 6. Train the Model

Train the XGBoost classifier with the current hyperparameter settings and the configured evaluation set.


In [ ]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()
X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy()

model = XGBClassifier(**hyperparameters)
model.fit(
    X_train_np,
    y_train_np,
    eval_set=[(X_test_np, y_test_np)],
    verbose=False,
)

model


## 7. Evaluate and Save

Score the holdout set, collect the expanded metrics, and append the run outputs to the CSV history files.


In [ ]:
probabilities = model.predict_proba(X_test_np)[:, 1]
predictions = (probabilities >= prediction_threshold).astype(int)
actuals = y_test_np

false_positives = int(((actuals == 0) & (predictions == 1)).sum())
false_negatives = int(((actuals == 1) & (predictions == 0)).sum())

metrics = {
    "model": MODEL_NAME,
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "prediction_threshold": prediction_threshold,
    "accuracy": accuracy_score(actuals, predictions),
    "precision": precision_score(actuals, predictions, zero_division=0),
    "recall": recall_score(actuals, predictions, zero_division=0),
    "f1_score": f1_score(actuals, predictions, zero_division=0),
    "roc_auc": roc_auc_score(actuals, probabilities),
    "false_positives": false_positives,
    "false_negatives": false_negatives,
    "hyperparameters_json": hyperparameters_json,
}

metrics_frame = pl.DataFrame([metrics])
results_frame = X_test.with_columns(
    [
        pl.Series("actual", actuals),
        pl.Series("predicted", predictions),
        pl.Series("probability", probabilities),
        pl.lit(MODEL_NAME).alias("model"),
        pl.lit(RUN_ID).alias("run_id"),
        pl.lit(RUN_TIMESTAMP).alias("run_timestamp"),
        pl.lit(prediction_threshold).alias("prediction_threshold"),
        pl.lit(hyperparameters_json).alias("hyperparameters_json"),
    ]
)

model.save_model(str(MODEL_FILE))
append_frame_to_csv(metrics_frame, METRICS_FILE)
append_frame_to_csv(results_frame, RESULTS_FILE)

print(f"model saved to: {MODEL_FILE}")
print(f"metrics appended to: {METRICS_FILE}")
print(f"results appended to: {RESULTS_FILE}")
metrics_frame


## 8. Preview Results

Review the current hyperparameter settings, the latest metric summary, and a sample of the appended results output.


In [ ]:
print(hyperparameters_frame)
print(metrics_frame)
results_frame.head(10)
